In [1]:
!pip install groq python-dotenv numpy tqdm datasets

In [2]:
from groq import Groq
from dotenv import load_dotenv
from datasets import load_dataset

import os
from tqdm import tqdm
import re
import random
import pprint

from typing import List, Dict, Any

load_dotenv()
random.seed(0)

client = Groq()
gsm8k_dataset = load_dataset("gsm8k", "main")

gsm8k_train = gsm8k_dataset["train"]
gsm8k_test  = gsm8k_dataset["test"]

In [3]:
def generate_response_using_Llama(
        prompt: str,
        model: str = "llama-3.1-8b-instant"
    ):
    try:
        chat_completion = client.chat.completions.create(
            messages=[
                {
                    "role": "system",
                    "content": "You are a helpful assistant that solves math problems."
                },
                {
                    "role": "user", 
                    "content": prompt
                }
            ],
            model=model,
            temperature=0.1, ### 수정해도 됩니다!
            stream=False
        )
        return chat_completion.choices[0].message.content
    
    except Exception as e:
        print(f"API call error: {str(e)}")
        return None

#### 응답 잘 나오는지 확인해보기

In [4]:
response = generate_response_using_Llama(
    prompt="Hello world!",
)
print(response)

Hello! It's nice to meet you. I'm here to help with any math problems you might have. What's on your mind? Do you have a specific problem you'd like me to solve, or would you like some help with a particular math concept?


#### GSM8K 데이터셋 확인해보기

In [5]:
print("[Question]")
for l in gsm8k_test['question'][0].split("."):
    print(l)
print("="*100)
print("[Answer]")
print(gsm8k_test['answer'][0])

[Question]
Janet’s ducks lay 16 eggs per day
 She eats three for breakfast every morning and bakes muffins for her friends every day with four
 She sells the remainder at the farmers' market daily for $2 per fresh duck egg
 How much in dollars does she make every day at the farmers' market?
[Answer]
Janet sells 16 - 3 - 4 = <<16-3-4=9>>9 duck eggs a day.
She makes 9 * 2 = $<<9*2=18>>18 every day at the farmer’s market.
#### 18


#### Util 함수들
- extract_final_answer: LLM의 응답을 parse하여 최종 결과만 추출 (정답과 비교하기 위해)
- run_benchmark_test: 벤치마크 테스트
- save_final_result: 결과물 제출을 위한 함수

In [6]:
def extract_final_answer(response: str):
    """
    정답 추출 로직 개선: 
    1. None 처리 및 예외 방지
    2. 숫자 파싱 정규식 강화 (화폐 단위 $, 콤마 등 처리)
    """
    if not response:
        return None

    # 정규식 공통 패턴: 콤마(,)가 포함된 숫자도 처리 가능하도록 개선
    # 예: 1,234.56 -> 1234.56
    num_pattern = r"[-+]?(?:\d{1,3}(?:,\d{3})*|\d+)(?:\.\d+)?"

    # [1순위] My Prompting 약속 포맷 [[숫자]]
    special_matches = re.findall(r"\[\[\s*(" + num_pattern + r")\s*\]\]", response)
    if special_matches:
        return special_matches[-1].replace(",", "")

    # [2순위] Direct/CoT용 명시적 선언 (Answer: 숫자)
    # 대소문자 무시, :, $ 기호 유연하게 처리
    explicit_regex = r"(?:Answer|result|answer|is)\s*:?\s*\$?\s*(" + num_pattern + r")"
    explicit_matches = re.findall(explicit_regex, response, re.IGNORECASE)
    if explicit_matches:
        return explicit_matches[-1].replace(",", "")

    # [3순위] 텍스트의 맨 마지막 숫자 (Fallback)
    # 텍스트 끝부분에 있는 숫자를 잡되, 날짜 같은 오탐을 줄이기 위해 단순화
    all_numbers = re.findall(num_pattern, response)
    if all_numbers:
        return all_numbers[-1].replace(",", "")

    return None

In [13]:
def run_benchmark_test(
        dataset,
        prompt: str,
        model: str = "llama-3.1-8b-instant",
        num_samples: int = 50,
        VERBOSE: bool = False
    ):
    correct = 0
    total   = 0
    results = []

    for i in tqdm(range(min(num_samples, len(dataset)))):
        question = dataset[i]["question"]
        correct_answer = float(re.findall(r'\d+(?:\.\d+)?', dataset[i]["answer"].split('####')[-1])[0])

        response = generate_response_using_Llama(
            prompt=prompt.format(question=question),
            model=model
        )

        if response:
            if VERBOSE:
                print("="*50)
                print(response)
                print("="*50)
            predicted_answer = extract_final_answer(response)

            if isinstance(predicted_answer, str):
                predicted_answer = float(predicted_answer.replace(",", ""))
            
            diff = abs(predicted_answer - correct_answer)
            is_correct = diff < 1e-5 if predicted_answer is not None else False
            
            if is_correct:
                correct += 1
            total += 1
            
            results.append({
                'question': question,
                'correct_answer': correct_answer,
                'predicted_answer': predicted_answer,
                'response': response,
                'correct': is_correct
            })

            if (i + 1) % 5 == 0:
                current_acc = correct/total if total > 0 else 0
                print(f"Progress: [{i+1}/{num_samples}]")
                print(f"Current Acc.: [{current_acc:.2%}]")

    return results, correct/total if total > 0 else 0

In [8]:
def save_final_result(results: List[Dict[str, Any]], accuracy: float, filename: str) -> None:
    result_str = f"====== ACCURACY: {accuracy} ======\n\n"
    result_str += f"[Details]\n"
    
    for idx, result in enumerate(results):
        result_str += f"Question {idx+1}: {result['question']}\n"
        result_str += f"Correct Answer: {result['correct_answer']}\n"
        result_str += f"Predicted Answer: {result['predicted_answer']}\n"
        result_str += f"Correct: {result['correct']}\n\n"
    
    with open(filename, "w", encoding="utf-8") as f:
        f.write(result_str)

#### Direct prompting with few-shot example

In [9]:
def construct_direct_prompt(num_examples: int = 3) -> str:
    train_dataset = gsm8k_train

    sampled_indices = random.sample(
        [i for i in range(len(train_dataset['question']))],
        num_examples
    )

    prompt = "Instruction:\nSolve the following mathematical question and generate ONLY the answer after a tag, 'Answer:' without any rationale.\n"

    for i in range(num_examples):
        cur_question = train_dataset['question'][i]
        cur_answer = train_dataset['answer'][i].split("####")[-1].strip()

        prompt += f"\n[Example {i+1}]\n"
        prompt += f"Question:\n{cur_question}\n"
        prompt += f"Answer:{cur_answer}\n"

    prompt += "\nQuestion:\n{question}\nAnswer:"

    return prompt

In [14]:
### 어떤 방식으로 저장되는지 확인해보세요!
PROMPT = construct_direct_prompt(3)
VERBOSE = False

results, accuracy = run_benchmark_test(
    dataset=gsm8k_test,
    prompt=PROMPT,
    VERBOSE=VERBOSE,
    num_samples=10
)
save_final_result(results, accuracy, "example.txt")

 50%|█████     | 5/10 [00:02<00:02,  1.85it/s]

Progress: [5/10]
Current Acc.: [60.00%]


100%|██████████| 10/10 [00:04<00:00,  2.12it/s]

Progress: [10/10]
Current Acc.: [70.00%]


In [20]:
# TODO: 0 shot, 3 shot, 5 shot direct prompting을 통해 벤치마크 테스트를 한 후, 각각 direct_prompting_{shot: int}.txt로 저장해주세요!
# 예시: shot이 5인 경우 direct_prompting_5.txt
# 항상 num_samples=50 입니다!

for shot in [0, 3, 5]:
    PROMPT = construct_direct_prompt(shot)
    results, accuracy = run_benchmark_test(
        dataset=gsm8k_test,
        prompt=PROMPT,
        VERBOSE=False,
        num_samples=50
    )
    save_final_result(results, accuracy, f"direct_prompting_{shot}.txt")

 10%|█         | 5/50 [00:01<00:17,  2.64it/s]

Progress: [5/50]
Current Acc.: [80.00%]


 20%|██        | 10/50 [00:03<00:15,  2.53it/s]

Progress: [10/50]
Current Acc.: [70.00%]


 30%|███       | 15/50 [00:05<00:13,  2.60it/s]

...

Progress: [40/50]
Current Acc.: [75.00%]


 90%|█████████ | 45/50 [00:42<00:10,  2.09s/it]

Progress: [45/50]
Current Acc.: [71.11%]


100%|██████████| 50/50 [00:44<00:00,  1.11it/s]

Progress: [50/50]
Current Acc.: [72.00%]


### Chain-of-Thought prompting with few-shot example
```text
[Question]
Janet’s ducks lay 16 eggs per day
 She eats three for breakfast every morning and bakes muffins for her friends every day with four
 She sells the remainder at the farmers' market daily for $2 per fresh duck egg
 How much in dollars does she make every day at the farmers' market?
====================================================================================================
[Answer]
Janet sells 16 - 3 - 4 = <<16-3-4=9>>9 duck eggs a day.
She makes 9 * 2 = $<<9*2=18>>18 every day at the farmer’s market.
#### 18
```

[Answer] 아래의 정답을 도출하는 과정을 예시로 달아주면 CoT의 few shot이 되겠죠?

In [16]:
def construct_CoT_prompt(num_examples: int = 3) -> str:
    train_dataset = gsm8k_train

    sampled_indices = random.sample(
        [i for i in range(len(train_dataset['question']))],
        num_examples
    )
    prompt = "Instruction:\n Solve the following mathematical question. Think step by step to arrive at the correct answer. Finally, answer the question in the format 'Answer: [result]'.\n" 
    

    for i in range(num_examples):
        idx = sampled_indices[i]
        cur_question = train_dataset['question'][idx]
        cur_answer = train_dataset['answer'][idx]
        cot_answer = cur_answer.replace("####", "\nAnswer:\n")
        prompt += f"Example {i+1}:\n"
        prompt += f"Question: {cur_question}\n"
        prompt += f"Reasoning: {cot_answer}\n\n"
        pass

    prompt += "\nQuestion:\n{question}\nAnswer:"

    return prompt

In [19]:
# TODO: 0 shot, 3 shot, 5 shot CoT prompting을 통해 벤치마크 테스트를 한 후, 각각 CoT_prompting_{shot: int}.txt로 저장해주세요!
# 예시: shot이 5인 경우 CoT_prompting_5.txt
# 항상 num_samples=50 입니다!

for shot in [0, 3, 5]:
    PROMPT = construct_CoT_prompt(shot)
    results, accuracy = run_benchmark_test(
        dataset=gsm8k_test,
        prompt=PROMPT,
        VERBOSE=False,
        num_samples=50
    )
    save_final_result(results, accuracy, f"CoT_prompting_{shot}.txt")

 10%|█         | 5/50 [00:02<00:23,  1.94it/s]

Progress: [5/50]
Current Acc.: [100.00%]


 20%|██        | 10/50 [00:04<00:20,  1.94it/s]

Progress: [10/50]
Current Acc.: [80.00%]


 30%|███       | 15/50 [00:11<00:41,  1.18s/it]

...

Progress: [40/50]
Current Acc.: [75.00%]


 90%|█████████ | 45/50 [00:26<00:02,  2.35it/s]

Progress: [45/50]
Current Acc.: [75.56%]


100%|██████████| 50/50 [00:29<00:00,  1.72it/s]

Progress: [50/50]
Current Acc.: [76.00%]


### Construct your prompt!!

목표: 본인만의 프롬프트를 통해 정답률을 더 끌어올려보기!
- gsm8k의 train 데이터셋에서 예시를 가져온 다음 (자유롭게!)
- 그 예시들에 대한 풀이 과정을 만들어주세요!
- 모든 것들이 자유입니다! Direct Prompting, CoT Prompting을 한 결과보다 정답률만 높으면 돼요.

In [ ]:
import random

def construct_my_prompt(num_examples: int = 5) -> str:
    train_dataset = gsm8k_train
    
    sorted_indices = sorted(
        range(len(train_dataset['question'])), 
        key=lambda i: len(train_dataset['question'][i])
    )
    
    sampled_indices = sorted_indices[:num_examples]


    prompt = (
        "Instruction:\n"
        "You are a math expert. Solve the problem step-by-step logicially.\n"
        "The last line of your response MUST be the final answer in double brackets: [[NUMBER]].\n"
        "Do not write anything after the brackets.\n\n"
    )

    for i in range(num_examples):
        idx = sampled_indices[i]
        cur_question = train_dataset['question'][idx]
        raw_answer = train_dataset['answer'][idx]
        
        parts = raw_answer.split("####")
        reasoning = parts[0].strip()
        final_num = parts[1].strip().replace(',', '')

        prompt += f"Question: {cur_question}\n"
        prompt += f"Answer: {reasoning}\n"
        prompt += f"Therefore, the final answer is [[{final_num}]].\n\n"

    if num_examples >= 5:
        prompt += "(Remember: End with [[NUMBER]].)\n\n"

    prompt += (
        "Question: {question}\n"
        "Answer:" 
    )

    return prompt

In [42]:
for shot in [0, 3, 5]:
    # 1. 프롬프트 생성
    PROMPT = construct_my_prompt(shot)
    
    # 2. 테스트 실행
    results, accuracy = run_benchmark_test(
        dataset=gsm8k_test,
        prompt=PROMPT,
        VERBOSE=False,
        num_samples=50
    )
    
    # 3. 결과 저장 (My_prompting_{shot}.txt)
    # save_final_result 함수 인자 순서는 정의해두신 것에 맞춰주세요.
    # (보통 results, accuracy, filename 순서거나, filename이 맨 앞일 수 있습니다)
    save_final_result(results, accuracy, f"My_prompting_{shot}.txt")

 10%|█         | 5/50 [00:02<00:24,  1.82it/s]

Progress: [5/50]
Current Acc.: [80.00%]


 20%|██        | 10/50 [00:05<00:22,  1.74it/s]

Progress: [10/50]
Current Acc.: [70.00%]


 30%|███       | 15/50 [00:15<00:56,  1.61s/it]

...

Progress: [40/50]
Current Acc.: [80.00%]


 90%|█████████ | 45/50 [00:45<00:02,  2.11it/s]

Progress: [45/50]
Current Acc.: [80.00%]


100%|██████████| 50/50 [00:48<00:00,  1.04it/s]

Progress: [50/50]
Current Acc.: [82.00%]


### 보고서 작성하기
#### 아래의 내용이 포함되면 됩니다!

1. Direct Prompting, CoT Prompting, My Prompting을 0 shot, 3 shot, 5 shot 정답률을 표로 보여주세요!
2. CoT Prompting이 Direct Prompting에 비해 왜 좋을 수 있는지에 대해서 서술해주세요!
3. 본인이 작성한 프롬프트 기법이 CoT에 비해서 왜 더 좋을 수 있는지에 대해서 설명해주세요!
4. 최종적으로, `PROMPTING.md`에 보고서를 작성해주세요!